In [33]:
import sidechainnet as scn
import torch
from torch.utils.data import DataLoader
import numpy as np
import requests
import textwrap
from IPython.display import display, HTML, display
import torch.nn as nn
import torch.nn.functional as F
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
from keras import layers
import matplotlib.pyplot as plt
import json

In [34]:
dataloaders = scn.load(batch_size=64, shuffle=True, scn_dir="./data/", num_workers=4, with_pytorch="dataloaders")

SidechainNet was loaded from ./data/sidechainnet_casp12_30.pkl.


In [35]:
def cycle_gen(dataloader):
    while True:
        for batch in dataloader:
            x = batch.seqs_int.numpy()
            y = np.nan_to_num(batch.angles.numpy(), nan=0.0)
            yield x, y


In [36]:
train_gen = cycle_gen(dataloaders['train'])
val_gen = cycle_gen(dataloaders['valid-30'])


In [37]:
def BiLSTMModel(embedding_dim=64, lstm_units=64, dense_units=128):
    model = keras.Sequential([
        keras.Input(shape=(None,)),
        layers.Embedding(input_dim=21, output_dim=embedding_dim, mask_zero=True),
        layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True)),
        layers.Dense(dense_units, activation='relu'),
        layers.TimeDistributed(layers.Dense(12, activation='tanh'))  # Predicting 3 angles per residue, use tanh to constrain angles between -1 and 1
    ])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    return model

In [38]:
embed_8 = BiLSTMModel(8, 64, 128)

embed_8.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0), loss='mse', metrics=['mse'])

history = embed_8.fit(
    train_gen,
    steps_per_epoch=len(dataloaders['train']),
    validation_data=val_gen,
    validation_steps=len(dataloaders['valid-30']),
    epochs=2,
    verbose=1
)


Epoch 1/2
402/402 ━━━━━━━━━━━━━━━━━━━━ 488s 1s/step - loss: 2.3857 - mse: 2.3836 - val_loss: 1.1058 - val_mse: 1.1058
Epoch 2/2
402/402 ━━━━━━━━━━━━━━━━━━━━ 484s 1s/step - loss: 2.1794 - mse: 2.1777 - val_loss: 1.0922 - val_mse: 1.0922
